In [1]:
!ollama pull kronos483/Llama-3.2-3B-PubMed:latest

/bin/bash: line 1: ollama: command not found


In [3]:
!nvidia-smi

Sat Aug 29 10:06:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!apt-get update -qq
!apt-get install -y zstd pciutils
!pip -q install pandas openpyxl requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 77 not upgraded.
Need to get 946 kB of archives.
After this operation, 3,276 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 946 

In [6]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

response = requests.get(
    "http://127.0.0.1:11434/api/tags"
)

print("Ollama status:", response.status_code)

Ollama status: 200


In [8]:
!ollama pull kronos483/Llama-3.2-3B-PubMed:latest

In [9]:
!ollama list

NAME                                    ID              SIZE      MODIFIED       
kronos483/Llama-3.2-3B-PubMed:latest    149126abd625    6.4 GB    11 seconds ago    


In [10]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_LLM_Benchmark.xlsx"
)

RESULT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_PubMedLlama_Results.xlsx"
)

print("Input exists:", INPUT_FILE.exists())

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Benchmark"
)

print("Number of records:", len(df))

Input exists: True
Number of records: 500


In [11]:
df["pubmed_output"] = ""
df["pubmed_raw"] = ""
df["pubmed_status"] = ""
df["pubmed_runtime_sec"] = ""

df.to_excel(
    RESULT_FILE,
    index=False
)

print("Created:")
print(RESULT_FILE)

Created:
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_PubMedLlama_Results.xlsx


In [12]:
import json
import time
import requests

MODEL = "kronos483/Llama-3.2-3B-PubMed:latest"

URL = "http://127.0.0.1:11434/api/generate"


SYSTEM_MESSAGE = """
You are a biomedical information extraction system.
Follow the user's extraction instructions exactly.
Return only the requested JSON output.
""".strip()


EXTRACTION_TEMPLATE = """
Extract ALL diseases, disorders, syndromes, symptoms, or clinical conditions
for which the drug is explicitly indicated in the following
Indications and Usage text.

Rules:

1. Read the entire supplied text.
2. Include every explicitly stated clinical indication.
3. Include symptoms or clinical conditions when explicitly indicated
   for treatment, prevention, management, or relief.
4. Preserve clinically meaningful disease subtypes and modifiers.
5. Use wording from the supplied text as closely as possible.
6. Do not intentionally normalize or generalize disease names.
7. Do not infer conditions that are not explicitly supported by the text.
8. Exclude conditions explicitly stated as NOT indicated or NOT recommended.
9. Exclude causative organisms when they merely describe the cause of infection.
10. Exclude mechanisms, procedures, dosing information,
    treatment regimens, and patient demographics.
11. Remove duplicate clinical conditions.

Return ONLY valid JSON:

{{"diseases":["condition 1","condition 2"]}}

If there are genuinely no indications:

{{"diseases":[]}}

INDICATIONS AND USAGE:

{indication_text}
""".strip()

In [13]:
def parse_output(raw_output):

    raw_output = raw_output.strip()

    try:
        parsed = json.loads(raw_output)

    except json.JSONDecodeError:

        start = raw_output.find("{")
        end = raw_output.rfind("}")

        if start == -1 or end == -1:
            raise ValueError(
                "No JSON object found in model response."
            )

        parsed = json.loads(
            raw_output[start:end + 1]
        )

    if "diseases" not in parsed:
        raise ValueError(
            "JSON does not contain 'diseases'."
        )

    diseases = parsed["diseases"]

    if not isinstance(diseases, list):
        raise ValueError(
            "'diseases' must be a list."
        )

    cleaned = []
    seen = set()

    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:
            seen.add(key)
            cleaned.append(disease)

    return cleaned

In [14]:
def extract_pubmed(indication_text):

    user_message = EXTRACTION_TEMPLATE.format(
        indication_text=indication_text
    )

    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        + SYSTEM_MESSAGE
        + "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        + user_message
        + "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "raw": True,
        "stream": False,

        "options": {
            "temperature": 0.1,
            "num_ctx": 4096,
            "num_predict": 1000,

            "stop": [
                "<|eot_id|>",
                "<|end_of_text|>"
            ]
        }
    }

    start_time = time.time()

    response = requests.post(
        URL,
        json=payload,
        timeout=240
    )

    response.raise_for_status()

    raw = response.json()["response"]

    diseases = parse_output(raw)

    runtime = round(
        time.time() - start_time,
        2
    )

    return diseases, raw, runtime

In [15]:
test_text = """
Metoprolol is indicated for the treatment of hypertension and angina pectoris.
"""

diseases, raw, runtime = extract_pubmed(
    test_text
)

print("RAW:")
print(raw)

print("\nPARSED:")
print(diseases)

print("\nRuntime:", runtime)

RAW:
{"diseases":["hypertension","angina pectoris"]}

PARSED:
['hypertension', 'angina pectoris']

Runtime: 89.78


In [16]:
for i in range(5):

    record_id = df.loc[i, "record_id"]
    indication_text = str(
        df.loc[i, "indication_text"]
    )

    print()
    print("=" * 70)
    print(f"RECORD {i + 1}/5")
    print("Record ID:", record_id)

    print()
    print("EXPERT GOLD:")
    print(
        df.loc[i, "gold_expert_mentions"]
    )

    try:

        diseases, raw, runtime = extract_pubmed(
            indication_text
        )

        df.at[
            i,
            "pubmed_output"
        ] = json.dumps(
            diseases,
            ensure_ascii=False
        )

        df.at[
            i,
            "pubmed_raw"
        ] = raw

        df.at[
            i,
            "pubmed_status"
        ] = (
            "OK"
            if len(diseases) > 0
            else "OK_EMPTY"
        )

        df.at[
            i,
            "pubmed_runtime_sec"
        ] = runtime

        print()
        print("PUBMED-LLAMA OUTPUT:")
        print(diseases)

        print(
            "Runtime:",
            runtime,
            "seconds"
        )

    except Exception as e:

        df.at[
            i,
            "pubmed_status"
        ] = "FAILED"

        df.at[
            i,
            "pubmed_raw"
        ] = str(e)

        print()
        print("ERROR:")
        print(e)


# Save the 5-record test
df.to_excel(
    RESULT_FILE,
    index=False
)

print()
print("Saved to:")
print(RESULT_FILE)


RECORD 1/5
Record ID: 10004

EXPERT GOLD:
Erosive Esophagitis; Gastroesophageal Reflux Disease; GERD; Zollinger Ellison Syndrome

PUBMED-LLAMA OUTPUT:
['Erosive Esophagitis', 'GERD', 'Pathological Hypersecretory Conditions', 'Zollinger Ellison Syndrome']
Runtime: 2.23 seconds

RECORD 2/5
Record ID: 10008

EXPERT GOLD:
Anxiety Disorders; anxiety disorder; Anxiety; Anxiety associated with depression; Panic Disorder; agoraphobia

PUBMED-LLAMA OUTPUT:
['Generalized Anxiety Disorder', 'Panic Disorder']
Runtime: 0.72 seconds

RECORD 3/5
Record ID: 10012

EXPERT GOLD:
ventricular fibrillation; ventricular tachycardia; VT; VF

ERROR:
No JSON object found in model response.

RECORD 4/5
Record ID: 10016

EXPERT GOLD:
dermatoses; plaque type psoriasis

PUBMED-LLAMA OUTPUT:
['psoriasis', 'dermatoses', 'corticosteroid responsive dermatoses', 'inflammatory dermatoses', 'intertriginous areas']
Runtime: 3.86 seconds

RECORD 5/5
Record ID: 10017

EXPERT GOLD:
acute, painful

PUBMED-LLAMA OUTPUT:
['acu

In [17]:
def extract_pubmed(indication_text):

    user_message = EXTRACTION_TEMPLATE.format(
        indication_text=indication_text
    )

    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        + SYSTEM_MESSAGE
        + "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        + user_message
        + "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "raw": True,
        "stream": False,

        "options": {
            "temperature": 0.1,
            "num_ctx": 4096,
            "num_predict": 1000,

            "stop": [
                "<|eot_id|>",
                "<|end_of_text|>"
            ]
        }
    }

    start_time = time.time()

    response = requests.post(
        URL,
        json=payload,
        timeout=240
    )

    response.raise_for_status()

    raw = response.json()["response"]

    runtime = round(
        time.time() - start_time,
        2
    )

    # Important:
    # return raw response even if parsing fails
    try:

        diseases = parse_output(raw)

        status = (
            "OK"
            if diseases
            else "OK_EMPTY"
        )

    except Exception:

        diseases = []
        status = "FORMAT_FAILED"

    return diseases, raw, runtime, status

In [18]:
diseases, raw, runtime, status = extract_pubmed(
    indication_text
)

In [19]:
df.at[i, "pubmed_output"] = json.dumps(
    diseases,
    ensure_ascii=False
)

df.at[i, "pubmed_raw"] = raw

df.at[i, "pubmed_status"] = status

df.at[i, "pubmed_runtime_sec"] = runtime

In [21]:
def run_all_pubmed():

    total = len(df)

    for i in range(total):

        record_id = df.loc[i, "record_id"]

        current_status = str(
            df.at[i, "pubmed_status"]
        ).strip()


        # --------------------------------------------------
        # Skip records already completed
        # --------------------------------------------------

        if current_status in [
            "OK",
            "OK_EMPTY",
            "FORMAT_FAILED"
        ]:

            print(
                f"[{i + 1}/{total}] "
                f"Record {record_id} - already completed"
            )

            continue


        indication_text = str(
            df.loc[i, "indication_text"]
        )


        print()
        print("=" * 70)
        print(
            f"[{i + 1}/{total}] "
            f"Record ID: {record_id}"
        )


        # --------------------------------------------------
        # Run PubMed-Llama
        # --------------------------------------------------

        try:

            diseases, raw, runtime, status = extract_pubmed(
                indication_text
            )


            df.at[
                i,
                "pubmed_output"
            ] = json.dumps(
                diseases,
                ensure_ascii=False
            )


            df.at[
                i,
                "pubmed_raw"
            ] = raw


            df.at[
                i,
                "pubmed_status"
            ] = status


            df.at[
                i,
                "pubmed_runtime_sec"
            ] = runtime


            print(
                "Output:",
                diseases
            )

            print(
                "Status:",
                status
            )

            print(
                "Runtime:",
                runtime,
                "seconds"
            )


        except Exception as e:

            # This means a real technical/API failure,
            # not merely bad JSON from the model.

            df.at[
                i,
                "pubmed_status"
            ] = "FAILED"

            df.at[
                i,
                "pubmed_raw"
            ] = str(e)


            print(
                "TECHNICAL ERROR:",
                e
            )


        # --------------------------------------------------
        # Save checkpoint every 20 records
        # --------------------------------------------------

        if (
            (i + 1) % 20 == 0
            or
            i == total - 1
        ):

            df.to_excel(
                RESULT_FILE,
                index=False
            )

            print(
                ">>> CHECKPOINT SAVED TO GOOGLE DRIVE"
            )


    # ------------------------------------------------------
    # Final save
    # ------------------------------------------------------

    df.to_excel(
        RESULT_FILE,
        index=False
    )


    print()
    print("=" * 70)
    print("PUBMED-LLAMA FINISHED")
    print("=" * 70)

    print(
        "Results saved to:"
    )

    print(
        RESULT_FILE
    )

In [22]:
run_all_pubmed()

[1/500] Record 10004 - already completed
[2/500] Record 10008 - already completed

[3/500] Record ID: 10012
Output: []
Status: FORMAT_FAILED
Runtime: 0.8 seconds
[4/500] Record 10016 - already completed
[5/500] Record 10017 - already completed

[6/500] Record ID: 1001
Output: ['Hypertension', 'Angina Pectoris', 'Myocardial Infarction']
Status: OK
Runtime: 0.91 seconds

[7/500] Record ID: 10140
Output: ['Major Depression']
Status: OK
Runtime: 0.45 seconds

[8/500] Record ID: 10146
Output: ['insomnia']
Status: OK
Runtime: 0.43 seconds

[9/500] Record ID: 10153
Output: ['anxiety', 'anxiety and tension', 'psychoneurosis', 'pruritus', 'chronic urticaria', 'atopic dermatoses', 'contact dermatoses', 'histamine mediated pruritus', 'organic disease states', 'allergic conditions']
Status: OK
Runtime: 2.07 seconds

[10/500] Record ID: 10157
Output: ["Parkinson's disease", "idiopathic Parkinson's disease"]
Status: OK
Runtime: 0.7 seconds

[11/500] Record ID: 10173
Output: []
Status: FORMAT_FAILED


In [23]:
print(
    df["pubmed_status"]
    .value_counts(dropna=False)
)

pubmed_status
OK               431
FORMAT_FAILED     69
Name: count, dtype: int64


In [24]:
df.to_excel(
    RESULT_FILE,
    index=False
)

print("Saved:", RESULT_FILE)

Saved: /content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_PubMedLlama_Results.xlsx


In [25]:
failed_pubmed = df[
    df["pubmed_status"] == "FORMAT_FAILED"
]

print("FORMAT_FAILED records:", len(failed_pubmed))

display(
    failed_pubmed[
        [
            "record_id",
            "gold_expert_mentions",
            "pubmed_raw"
        ]
    ].head(15)
)

FORMAT_FAILED records: 69


,record_id,gold_expert_mentions,pubmed_raw
2,10012,ventricular fibrillation; ventricular tachycar...,"{""diseases"":[""Ventricular fibrillation"",""Ventr..."
10,10173,partial onset seizures; epilepsy; myoclonic se...,"{""diseases"":[""partial onset seizures"",""myoclon..."
17,10312,type 2 diabetes mellitus,"{""diseases"":[""type 2 diabetes mellitus"",""diabe..."
40,1000,Rocky mountain spotted fever; typhus fever; ty...,"{""diseases"":[""Rocky mountain spotted fever"",""t..."
45,10029,"Infections of the ear, nose, and throat; Infec...","{""diseases"":[""Gonorrhea"",""Duodenal Ulcer"",""Pyl..."
54,10072,Attention Deficit Hyperactivity Disorder; ADHD,"{""diseases"":[""Attention Deficit Hyperactivity ..."
55,10073,muscle spasm,"{""diseases"":[""muscle spasm"",""musculoskeletal c..."
56,10082,pneumonia; chronic obstructive pulmonary disea...,"{""diseases"":[""Acute bacterial exacerbations of..."
60,10090,"severe, acute pain; severe acute pain; breakth...","{""diseases"":[""Acute pain"",""Moderately severe a..."
64,10099,depression; anxiety,"{""diseases"":[""Major Depressive Episode"",""Depre..."


In [26]:
for i, row in failed_pubmed.head(10).iterrows():

    print("=" * 80)
    print("RECORD ID:", row["record_id"])
    print()
    print("RAW RESPONSE:")
    print(repr(row["pubmed_raw"]))
    print()

RECORD ID: 10012

RAW RESPONSE:
'{"diseases":["Ventricular fibrillation","Ventricular tachycardia"]'

RECORD ID: 10173

RAW RESPONSE:
'{"diseases":["partial onset seizures","myoclonic seizures","primary generalized tonic clonic seizures","juvenile myoclonic epilepsy","idiopathic generalized epilepsy"]'

RECORD ID: 10312

RAW RESPONSE:
'{"diseases":["type 2 diabetes mellitus","diabetic ketoacidosis"]'

RECORD ID: 1000

RAW RESPONSE:
'{"diseases":["Rocky mountain spotted fever","typhus fever","typhus group","Q fever","rickettsialpox","tick fevers","Mycoplasma pneumoniae respiratory tract infections","Lymphogranuloma venereum","Psittacosis","trachoma","inclusion conjunctivitis","uncomplicated urethral infections","endocervical infections","rectal infections","nongonococcal urethritis","Relapsing fever","Chancroid","plague","Tularemia","Cholera","Campylobacter fetus infections","Brucellosis","Bartonellosis","Granuloma inguinale","Escherichia coli infections","Enterobacter aerogenes infecti

In [27]:
!pip -q install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.3 MB/s eta 0:00:00


In [28]:
import json
from json_repair import repair_json


def repair_pubmed_output(raw):

    raw = str(raw).strip()

    # --------------------------------------------
    # 1. No structured JSON at all
    # --------------------------------------------
    if not raw.startswith('{"diseases"'):

        return [], "FORMAT_FAILED"


    # --------------------------------------------
    # 2. Clearly truncated in the middle
    #
    # A complete response should normally finish
    # with ] or }.
    # --------------------------------------------
    if not (
        raw.endswith("]")
        or
        raw.endswith("}")
    ):

        return [], "FORMAT_FAILED_TRUNCATED"


    # --------------------------------------------
    # 3. Try normal JSON first
    # --------------------------------------------
    try:

        parsed = json.loads(raw)

    except Exception:

        # ----------------------------------------
        # 4. Deterministic JSON syntax repair
        # ----------------------------------------
        try:

            repaired = repair_json(raw)

            parsed = json.loads(repaired)

        except Exception:

            return [], "FORMAT_FAILED"


    diseases = parsed.get("diseases", [])


    if not isinstance(diseases, list):

        return [], "FORMAT_FAILED"


    # --------------------------------------------
    # Remove duplicate strings only
    # --------------------------------------------

    cleaned = []
    seen = set()

    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:

            seen.add(key)
            cleaned.append(disease)


    return cleaned, "OK_REPAIRED"

In [29]:
failed_indices = df.index[
    df["pubmed_status"] == "FORMAT_FAILED"
].tolist()


for i in failed_indices:

    raw = df.at[i, "pubmed_raw"]

    diseases, new_status = repair_pubmed_output(raw)


    df.at[
        i,
        "pubmed_status"
    ] = new_status


    df.at[
        i,
        "pubmed_output"
    ] = json.dumps(
        diseases,
        ensure_ascii=False
    )


# Save
df.to_excel(
    RESULT_FILE,
    index=False
)

print("Updated file saved.")

Updated file saved.


In [30]:
print(
    df["pubmed_status"]
    .value_counts(dropna=False)
)

pubmed_status
OK                         431
OK_REPAIRED                 57
FORMAT_FAILED_TRUNCATED     12
Name: count, dtype: int64


In [31]:
df.to_excel(
    RESULT_FILE,
    index=False
)

print("Saved:", RESULT_FILE)

Saved: /content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_PubMedLlama_Results.xlsx
